In [1]:
import pandas as pd
import time
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.output_parsers import JsonOutputParser
from langchain.output_parsers import RetryOutputParser
from pydantic import BaseModel, Field
from langchain_aws import ChatBedrockConverse
from typing import List, Dict

In [2]:
jd = pd.read_excel("data/two_jobs_ads.xlsx")

In [3]:
job_names = []
jd_dict = []
for _, row in jd.iterrows():
    job_names.append(row["Role Title"])
    jd_dict.append(row.to_dict())

In [5]:
fc = pd.read_excel("data/two_jobs_applications.xlsx")
fc = fc[fc["Item"].str.contains("Personality Choice")]
fc_dict = []
for job in job_names:
    job_df = fc[fc["Job"] == job]
    fc_dict.append({i: q[209:].split(" | ") for i, q in zip(range(len(job_df.index)), job_df["Question"])})

In [8]:
from enum import Enum

class Option(str, Enum):
    A = "optionA"
    B = "optionB"
    NEITHER = "NEITHER"
    
class Answer(BaseModel):
    answer: Option
    rationale_positive: str
    rationale_negative: str

class FC(BaseModel):
    required: Dict[int, Answer]

In [9]:
llm = ChatBedrockConverse(
    model="anthropic.claude-3-5-haiku-20241022-v1:0",
    temperature=0,
    max_tokens=8192,
)

In [10]:
json_parser = JsonOutputParser(pydantic_object=FC)
retry_parser = RetryOutputParser.from_llm(parser=json_parser, llm=llm, max_retries=3)

In [11]:
with open("prompt/forced_choice_prompt.txt") as f:
    prompt_text = f.read()

In [12]:
format_instructions = json_parser.get_format_instructions()
prompt = PromptTemplate(
    template=prompt_text,
    input_variables=["job_description", "personality"],
    partial_variables={"format_instructions": format_instructions},
)

In [13]:
chain = RunnableParallel(
    completion=prompt | llm, 
    prompt_value=prompt
) | RunnableLambda(lambda x: retry_parser.parse_with_prompt(x["completion"].content, x["prompt_value"]))

In [20]:
required = {}
start_time = time.time()
start_idx = 0
for i in range(len(jd_dict) + 1):
    if (i % 10 == 0 and i != 0) or i == len(jd_dict):
        batch_input = [{
            "job_description": jd_dict[i],
            "personality_pairs": fc_dict[i]
        } for i in range(start_idx, i)]
        response = chain.batch(batch_input)
        for j, r in zip(range(start_idx, i), response):
            required[job_names[j]] = r["required"]

        time_lapse = time.time() - start_time
        print(start_idx, i, time_lapse)
        start_idx = i
        if time_lapse < 60 and i != len(jd_dict):
            time.sleep(60 - time_lapse)
        start_time = time.time()

0 2 48.45195412635803


In [18]:
len(required)

2

In [21]:
import json
with open("intermediate_data/personality_fc.json", "w") as f:
    json.dump(required, f)